In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


spark = (
        SparkSession
            .builder
            .master("local[*]")
            .appName("notebook_teste")
            .config(
                "spark.jars.packages",
                "org.apache.hadoop:hadoop-aws:3.3.4,"
                "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
                "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1,"
                "org.scala-lang:scala-library:2.13.12")
            # Configuração do Catálogo Iceberg
            .config("spark.sql.catalog.icebergtb", "org.apache.iceberg.spark.SparkCatalog")
            .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties") 
            .config("spark.log.level", "ERROR") 
            .config("spark.eventLog.enabled", "false") 
            # .config("spark.sql.catalog.icebergtb.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
            .config("spark.sql.catalog.icebergtb.type", "hadoop") 
            .config("spark.sql.catalog.icebergtb.warehouse", "s3a://iceberg/")
            .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.endpoint", "http://172.19.192.1:9000")
            .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.access.key", "minioadmin")
            .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.secret.key", "minioadmin")
            .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.path.style.access", "true")
            .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.connection.ssl.enabled", "false")
            .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
            .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
        .getOrCreate()
)

spark

In [ ]:
dataframe = spark.sql(""" select * from icebergtb.advworks_rdb.raw_dimsalesperson """)

In [ ]:
dataframe = (dataframe
                .selectExpr('*',
                            
                            """ 
                            case
                                when gender = 'M' then 'Male'
                                else 'Female'
                            end as gender_new
                            """
                ))

In [ ]:
dataframe = (dataframe.withColumn(
                        "territoryid_new",
                        when(
                            (col("TerritoryID").isNull()) & (col("CountryRegionCode").isNull()) & (col("JobTitle").like("North American%")),
                            lit(1))
                        .when(
                            (col("TerritoryID").isNull()) & (col("CountryRegionCode").isNull()) & (col("JobTitle").like("Pacific Sales%")),
                            lit(9))
                        .when(
                            (col("TerritoryID").isNull()) & (col("CountryRegionCode").isNull()) & (col("JobTitle").like("European Sales%")),
                            lit(8))
                        .otherwise(col("TerritoryID"))
                    ))

In [ ]:
dataframe = (dataframe.withColumn(
                        "countrycode_new",
                        when(
                            (col("TerritoryID").isNull()) & (col("JobTitle").like("North American%")),
                            lit("US"))
                        .when(
                            (col("TerritoryID").isNull()) & (col("JobTitle").like("European Sales%")),
                            lit("EU"))
                        .when(
                            (col("TerritoryID").isNull()) & (col("JobTitle").like("Pacific Sales%")),
                            lit("UA"))
                        .otherwise(col("CountryRegionCode"))
                    ))

In [ ]:
dataframe = (dataframe.withColumn(
                        "salesregion_new",
                        when(
                            (col("JobTitle").like("North American%")) & (col("ContinentName").isNull()),
                            lit("USA HQ"))
                        .when(
                            (col("JobTitle").like("Pacific Sales%")) & (col("ContinentName").isNull()),
                            lit("Sidney HQ"))
                        .when(
                            (col("JobTitle").like("European Sales%")) & (col("ContinentName").isNull()),
                            lit("Berlim HQ"))
                        .otherwise(col("ContinentName"))
                    ))

In [ ]:
max_sales_quota = dataframe.selectExpr("cast(max(salesquota) as float)").collect()[0][0]

dataframe = (dataframe.withColumn(
                        "salesquota_new",
                        when((col("jobtitle").like("North American%")) & (col("salesquota").isNull()),
                                lit(max_sales_quota * 0.70 + max_sales_quota))

                        .when((col("jobtitle").like("European Sales%")) & (col("salesquota").isNull()),
                                lit(max_sales_quota * 0.65 + max_sales_quota))

                        .when((col("jobtitle").like("Pacific Sales%")) & (col("salesquota").isNull()),
                                lit(max_sales_quota * 0.55 + max_sales_quota))

                        .otherwise(col("salesquota"))
                    ))

In [ ]:
dataframe = (dataframe .withColumn(
                        "commission_new",
                        
                         when((col("jobtitle").like('North American%')) & (col("commissionpct")== 0.00), lit(0.05))                
                        .when((col("jobtitle").like('European Sales%')) & (col("commissionpct")== 0.00), lit(0.04))                
                        .when((col("jobtitle").like('Pacific Sales%')) & (col("commissionpct")== 0.00), lit(0.04))
                    ))

In [ ]:
dataframe = dataframe.selectExpr("*",
                                 
                                """ 
                                case
                                    when SalariedFlag = 1 then 'Yes'
                                    else 'No'
                                end as issalaried
                                
                                """,
                                
                                """
                                case 
                                    when CurrentFlag = 1 then 'Yes'
                                    else 'No'
                                end as isactive
                                
                                """
                            )

In [ ]:
dataframe = (dataframe
                .withColumn("age", year(current_date()) - year(col("birthdate"))) 
                .withColumn("yearsincompany", year(current_date()) - year(col("hiredate")))
            )


In [ ]:
dataframe.printSchema()